# Adapters

**Tags:** parameter-efficient, finetuning
**Prerequisites:** Fine-tuning
**Related Concepts:** See flowchart below
**Source:** llm/concepts/adapters.md

## TL;DR

Bottleneck modules (down-project → activation → up-project) inserted into transformer layers. Update only adapters, freeze base weights. 2-5% parameter overhead; achieves 96-98% of full FT quality. Alternative to LoRA with different architecture; can combine multiple adapters per task.

## Core Intuition

Like LoRA, adapters train small modules instead of full weights. But instead of low-rank matrix updates (AB^T), adapters use bottleneck feed-forward layers. Similar efficiency (1-3% params), different structure. Useful when you want architectural flexibility or specific layer patterns.

## How It Works

**Adapter Architecture:**
```
Input (hidden state h ∈ ℝ^d)
  ↓
Down-project: h_down = W_down @ h + b_down  (d → r)
  ↓
Activation: h_act = ReLU(h_down)
  ↓
Up-project: h_out = W_up @ h_act + b_up  (r → d)
  ↓
Output: y = h + α * h_out  (residual connection)
```

**Bottleneck Design:**
- Input dimension: d (hidden size, typically 768 or 1024)
- Bottleneck dimension: r (adapter size, typically 32-64)
- Trainable parameters: r*d + d*r + 2d ≈ 2*r*d
- For d=768, r=64: ~98k params per adapter
- Multiple adapters can be stacked: series, parallel, or hierarchical

**Placement in Transformer:**
Adapters typically inserted after attention or FF layer in each transformer block:
```
MultiHeadAttention(x) + x
    ↓
Adapter(x)  [trained]
    ↓
LayerNorm
    ↓
FeedForward(x) + x
    ↓
Adapter(x)  [trained]
```

**Multi-Task Adaptation:**
- Single-adapter: one adapter for one task
- Stacked-adapters: layers of adapters for increasing capacity
- Parallel-adapters: different adapters for different modalities or domains
- Adapter fusion: learnable combination of task-specific adapters

### Workflow Diagram

```mermaid
graph LR
    A["Input"] --> B["Adapters Process"]
    B --> C["Output"]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#e8f5e9
```

**Note:** This is a basic workflow template. Review and customize based on specific concept.

## Key Properties & Trade-offs

/ Trade-offs

| Property | Adapters | LoRA | Full FT | Prefix Tuning |
|----------|----------|------|---------|--------------|
| Trainable params | 2-5% | 1-3% | 100% | 1-2% |
| Flexibility | High (add/remove) | High | N/A | Medium |
| Architecture | Bottleneck | Low-rank | All params | Prefix only |
| Multi-task | Yes (stack/parallel) | Yes (multi-LoRA) | No | Limited |
| Inference latency | +5-10% | Merged (0%) | 1x | +2-5% |
| Memory during training | 80-90% saved | 90-95% saved | None | 85-90% saved |

**When to use adapters:**
- Multiple tasks on same base (stack adapters per task)
- Domain-specific fine-tuning with flexibility
- When you prefer architectural over rank-based parametrization
- If you need to add/remove skills without retraining

**When LoRA is better:**
- Merging for deployment is important (LoRA merges clean into weights)
- You want minimal inference overhead
- Simplicity is priority

### Code Implementation

```python
# Key imports for Adapters
import numpy as np
import torch
from typing import Any

# Adapters example implementation
class Adapters:
    """
    Adapters implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts

```mermaid
graph TD
    A["Adapters"]
    B["Fine-tuning"] -->|prerequisite| A
    A -->|used with| D["LoRA (Low-Rank Adaptation)"]
    A -->|used with| D["Prefix Tuning"]
    
    style A fill:#fff3e0
```

### Common Interview Questions

**Q: What is Adapters used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Adapters?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Adapters compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Adapters?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Adapters?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/adapters.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]